In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from torch.utils.data import TensorDataset, DataLoader

PROJECT_ROOT = Path('/home/vlad/Funcional-genetic-element-recognition-agent')
DATASET_PATH = PROJECT_ROOT / 'data_code' / 'processed' / 'dataset.pt'

data = torch.load(DATASET_PATH, weights_only=False)
CLASSES = data['classes']
NUM_CLASSES = len(CLASSES)
HIDDEN_DIM = data['train_X'].shape[1]
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {device}")
print(f"Classes: {CLASSES}")
print(f"Train: {data['train_X'].shape}, Val: {data['val_X'].shape}")
print(f"Holdout аксессионы: {list(data['holdout'].keys())}")
print(f"Config: {data['config']}")

In [ ]:
print("Train per-class:")
for i, cls in enumerate(CLASSES):
    n = (data['train_y'] == i).sum().item()
    print(f"  {cls}: {n} ({100*n/len(data['train_y']):.1f}%)")
print("\nVal per-class:")
for i, cls in enumerate(CLASSES):
    n = (data['val_y'] == i).sum().item()
    print(f"  {cls}: {n} ({100*n/len(data['val_y']):.1f}%)")
print("\nHoldout per-class:")
for acc, h in data['holdout'].items():
    dist = {cls: int((h['y'] == i).sum()) for i, cls in enumerate(CLASSES)}
    print(f"  {acc}: total={len(h['y'])}  {dist}")

In [ ]:
class BioSequenceClassifier(nn.Module):
    def __init__(self, hidden_dim=HIDDEN_DIM, num_classes=NUM_CLASSES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
counts = torch.bincount(data['train_y'], minlength=NUM_CLASSES).float()
class_weights = counts.sum() / (counts + 1e-8)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES

print("Class weights:", {cls: round(class_weights[i].item(), 3) for i, cls in enumerate(CLASSES)})

train_loader = DataLoader(TensorDataset(data['train_X'], data['train_y']),
                          batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(data['val_X'], data['val_y']),
                        batch_size=128)

model = BioSequenceClassifier().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

In [ ]:
EPOCHS = 30
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    model.train()
    tloss, n = 0.0, 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        tloss += loss.item() * X.size(0)
        n += X.size(0)

    model.eval()
    vloss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            vloss += criterion(out, y).item() * X.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)

    history['train_loss'].append(tloss / n)
    history['val_loss'].append(vloss / total)
    history['val_acc'].append(100 * correct / total)

    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f"Ep {epoch+1:02d}/{EPOCHS} | train {history['train_loss'][-1]:.4f} | "
              f"val {history['val_loss'][-1]:.4f} | acc {history['val_acc'][-1]:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['val_acc'])
axes[1].set_title('Val accuracy (%)'); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in val_loader:
        out = model(X.to(device))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(y.numpy())

cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted'); plt.ylabel('Real')
plt.title('Validation confusion matrix')
plt.show()

print(classification_report(all_labels, all_preds,
                            labels=list(range(NUM_CLASSES)),
                            target_names=CLASSES, zero_division=0))

In [ ]:
holdout_preds = {}
model.eval()
for acc, h in data['holdout'].items():
    X = h['X'].to(device)
    y = h['y'].numpy()
    with torch.no_grad():
        out = model(X)
        probs = torch.softmax(out, dim=1).cpu().numpy()
        preds = out.argmax(1).cpu().numpy()
    holdout_preds[acc] = {'preds': preds, 'probs': probs}

    print(f"\n=== {acc} ({len(y)} окон) ===")
    print(classification_report(y, preds,
                                labels=list(range(NUM_CLASSES)),
                                target_names=CLASSES, zero_division=0))
    cm = confusion_matrix(y, preds, labels=list(range(NUM_CLASSES)))
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES)
    plt.xlabel('Predicted'); plt.ylabel('Real')
    plt.title(f'{acc}: confusion matrix (holdout)')
    plt.show()

In [ ]:
PROB_COLORS = {'INT': 'tab:blue', 'RT': 'tab:orange', 'RN': 'tab:green',
               'NEG': 'lightgray'}

for acc, h in data['holdout'].items():
    pos = h['positions'].numpy()
    y_true = h['y'].numpy()
    preds = holdout_preds[acc]['preds']
    probs = holdout_preds[acc]['probs']

    fig, axes = plt.subplots(NUM_CLASSES, 1, figsize=(16, 2.2 * NUM_CLASSES), sharex=True)
    for i, cls in enumerate(CLASSES):
        ax = axes[i]
        ax.fill_between(pos, 0, probs[:, i], color=PROB_COLORS.get(cls, 'gray'),
                        alpha=0.4, label=f'P({cls})')
        true_mask = y_true == i
        pred_mask = preds == i
        if true_mask.any():
            ax.scatter(pos[true_mask], np.full(true_mask.sum(), 1.05),
                       c='green', s=8, marker='|', label=f'true {cls}')
        if pred_mask.any():
            ax.scatter(pos[pred_mask], np.full(pred_mask.sum(), 1.15),
                       c='red', s=8, marker='|', label=f'pred {cls}')
        ax.set_ylabel(cls)
        ax.set_ylim(-0.05, 1.25)
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(alpha=0.3)
    axes[-1].set_xlabel('Position (bp)')
    plt.suptitle(f'Holdout {acc}: реальная разметка vs предсказания')
    plt.tight_layout()
    plt.show()